In [ ]:
!pip install -U datasets transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 12.6 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
  Attempting uninstall: datasets
    Found existing installation: datasets 2.14.4
    Uninstalling datasets-2.14.4:
      Successfully uninstalled datasets-2.14.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; pla

from datasets import load_dataset

- hf_ds = load_dataset(path="hfpath", name="hf_ds_name")
- csv_ds = load_dataset("csv", data_files="myfile.csv")
- text_ds = load_dataset("text", data_files="myfile.txt")
- json_ds = load_dataset("json", data_files="myfile.json")
- pkl_ds = load_dataset("pandas", data_files="mydf.pkl")

### Loading a local dataset
For this, we’ll use some drug effectiveness dataset from external link.

In [ ]:
!wget -P . "https://archive.ics.uci.edu/ml/machine-learning-databases/00462/drugsCom_raw.zip"

--2025-06-21 09:08:22--  https://archive.ics.uci.edu/ml/machine-learning-databases/00462/drugsCom_raw.zip
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified
Saving to: ‘./drugsCom_raw.zip’

drugsCom_raw.zip        [       <=>          ]  41.00M  31.8MB/s    in 1.3s    

2025-06-21 09:08:24 (31.8 MB/s) - ‘./drugsCom_raw.zip’ saved [42989872]



In [ ]:
import zipfile

with zipfile.ZipFile("drugsCom_raw.zip", "r") as zipref:
    zipref.extractall()

Since TSV is just a variant of CSV that uses tabs instead of commas as the separator, we can load these files by using the csv loading script and specifying the delimiter argument in the load_dataset() function as follows.

In [ ]:
import datasets
from datasets import load_dataset

ds = load_dataset("csv", data_files="drugsComTrain_raw.tsv", delimiter="\t")
ds

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount'],
        num_rows: 161297
    })
})

In [ ]:
# Peek at first few samples
ds_sample = ds["train"].shuffle(seed=42).select(range(1000))
ds_sample

Dataset({
    features: ['Unnamed: 0', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount'],
    num_rows: 1000
})

In [ ]:
ds_sample[0]

{'Unnamed: 0': 87571,
 'drugName': 'Naproxen',
 'condition': 'Gout, Acute',
 'review': '"like the previous person mention, I&#039;m a strong believer of aleve, it works faster for my gout than the prescription meds I take. No more going to the doctor for refills.....Aleve works!"',
 'rating': 9.0,
 'date': 'September 2, 2015',
 'usefulCount': 36}

In [ ]:
ds_sample[:2]

{'Unnamed: 0': [87571, 178045],
 'drugName': ['Naproxen', 'Duloxetine'],
 'condition': ['Gout, Acute', 'ibromyalgia'],
 'review': ['"like the previous person mention, I&#039;m a strong believer of aleve, it works faster for my gout than the prescription meds I take. No more going to the doctor for refills.....Aleve works!"',
  '"I have taken Cymbalta for about a year and a half for fibromyalgia pain. It is great\r\nas a pain reducer and an anti-depressant, however, the side effects outweighed \r\nany benefit I got from it. I had trouble with restlessness, being tired constantly,\r\ndizziness, dry mouth, numbness and tingling in my feet, and horrible sweating. I am\r\nbeing weaned off of it now. Went from 60 mg to 30mg and now to 15 mg. I will be\r\noff completely in about a week. The fibro pain is coming back, but I would rather deal with it than the side effects."'],
 'rating': [9.0, 3.0],
 'date': ['September 2, 2015', 'November 7, 2011'],
 'usefulCount': [36, 13]}

Great, we’ve loaded our first local dataset! But while this worked for the training set, what we really want is to include both the train and test splits in a single DatasetDict object so we can apply Dataset.map() functions across both splits at once. To do this, we can provide a dictionary to the data_files argument that maps each split name to a file associated with that split.

In [ ]:
ds = load_dataset("csv", data_files={"train": "drugsComTrain_raw.tsv", "test": "drugsComTest_raw.tsv"}, delimiter="\t")
ds

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount'],
        num_rows: 161297
    })
    test: Dataset({
        features: ['Unnamed: 0', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount'],
        num_rows: 53766
    })
})

### Streaming Datasets

To enable dataset streaming you just need to pass the `streaming=True` argument to the `load_dataset()` function. Streaming mode will make sure your PC won't have to load the whole dataset at once.

In [ ]:
ds_streamed = load_dataset("csv",
                           data_files={"train": "drugsComTrain_raw.tsv", "test": "drugsComTest_raw.tsv"},
                           delimiter="\t",
                           streaming=True)

In [ ]:
ds_streamed

IterableDatasetDict({
    train: IterableDataset({
        features: ['Unnamed: 0', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount'],
        num_shards: 1
    })
    test: IterableDataset({
        features: ['Unnamed: 0', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount'],
        num_shards: 1
    })
})

In [ ]:
type(ds_streamed)

datasets.dataset_dict.IterableDatasetDict

As the name suggests, to access the elements of an IterableDataset we need to iterate over it. We can access the first element of our streamed dataset as follows.

In [ ]:
next(iter(ds_streamed["train"]))

{'Unnamed: 0': 206461,
 'drugName': 'Valsartan',
 'condition': 'Left Ventricular Dysfunction',
 'review': '"It has no side effect, I take it in combination of Bystolic 5 Mg and Fish Oil"',
 'rating': 9.0,
 'date': 'May 20, 2012',
 'usefulCount': 27}

The elements from a streamed dataset can be processed on the fly using `IterableDataset.map()`, which is useful during training if you need to tokenize the inputs.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
tokenizes_ds = ds_streamed.map(lambda x: tokenizer(x["review"]))

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
next(iter(tokenizes_ds["train"]))

{'Unnamed: 0': 206461,
 'drugName': 'Valsartan',
 'condition': 'Left Ventricular Dysfunction',
 'review': '"It has no side effect, I take it in combination of Bystolic 5 Mg and Fish Oil"',
 'rating': 9.0,
 'date': 'May 20, 2012',
 'usefulCount': 27,
 'input_ids': [101,
  1000,
  2009,
  2038,
  2053,
  2217,
  3466,
  1010,
  1045,
  2202,
  2009,
  1999,
  5257,
  1997,
  2011,
  16033,
  10415,
  1019,
  11460,
  1998,
  3869,
  3514,
  1000,
  102],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1]}

We can also shuffle a streamed dataset using `IterableDataset.shuffle()`, but unlike `Dataset.shuffle()`, this only shuffles the elements in a predefined `buffer_size`.

In [ ]:
shuffle_ds_streamed = ds_streamed.shuffle(buffer_size=10_000, seed=42)

In [ ]:
next(iter(shuffle_ds_streamed["train"]))

{'Unnamed: 0': 175494,
 'drugName': 'Scopolamine',
 'condition': 'Motion Sickness',
 'review': '"The reason I&#039;m giving it a 9 instead of 10 is because of it&#039;s side effects. Otherwise, this patch works really well for an overnight or two day fishing trip (at least in my experience). I thought I&#039;m getting cold and feeling sick after I get off the fishing boat because my throat was getting dry and dryer and dryer. At that time I&#039;m not aware that one of it&#039;s side effects was dry mouth/throat. Aside from that, I didn&#039;t feel anything anymore after couple of days. I&#039;ll use it again for my next fishing trip. I&#039;ll opt for a dry throat verses getting sick on the boat."',
 'rating': 9.0,
 'date': 'August 24, 2010',
 'usefulCount': 9}

In this example, we selected a random example from the first 10,000 examples in the buffer. Once an example is accessed, its spot in the buffer is filled with the next example in the corpus (i.e., the 10,001st example in the case above). You can also select elements from a streamed dataset using the `IterableDataset.take()` and `IterableDataset.skip()` functions, which act in a similar way to `Dataset.select()`. For example, to select the first 5 examples in the dataset we can do the following

In [ ]:
ds_head = ds_streamed["train"].take(5)
ds_head

IterableDataset({
    features: ['Unnamed: 0', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount'],
    num_shards: 1
})

In [ ]:
list(ds_head)

[{'Unnamed: 0': 206461,
  'drugName': 'Valsartan',
  'condition': 'Left Ventricular Dysfunction',
  'review': '"It has no side effect, I take it in combination of Bystolic 5 Mg and Fish Oil"',
  'rating': 9.0,
  'date': 'May 20, 2012',
  'usefulCount': 27},
 {'Unnamed: 0': 95260,
  'drugName': 'Guanfacine',
  'condition': 'ADHD',
  'review': '"My son is halfway through his fourth week of Intuniv. We became concerned when he began this last week, when he started taking the highest dose he will be on. For two days, he could hardly get out of bed, was very cranky, and slept for nearly 8 hours on a drive home from school vacation (very unusual for him.) I called his doctor on Monday morning and she said to stick it out a few days. See how he did at school, and with getting up in the morning. The last two days have been problem free. He is MUCH more agreeable than ever. He is less emotional (a good thing), less cranky. He is remembering all the things he should. Overall his behavior is bett

In [ ]:
# Skip the first 1,000 examples and include the rest in the training set
train_ds = shuffle_ds_streamed["train"].skip(1000)
# Take the first 1,000 examples for the validation set
val_ds = shuffle_ds_streamed["train"].take(1000)

For preprocessing part, we shall use the non-streamed version, since our data is not that big anyway. But remember to use streaming when your data is like >5GB..

### 2. Preprocessing
From the samples we can already see a few quirks in our dataset:

- The Unnamed: 0 column looks suspiciously like an anonymized ID for each patient.
- The condition column includes a mix of uppercase and lowercase labels.
- The reviews are of varying length and contain a mix of Python line separators (\r\n) as well as HTML character codes like `&#039;`.

Let’s see how we can use Datasets to deal with each of these issues.

#### Rename patient id

 To test the patient ID hypothesis for the Unnamed: 0 column, we can use the Dataset.unique() function to verify that the number of IDs matches the number of rows in each split:

In [ ]:
for split in ds.keys():
    print(split)
    assert len(ds[split]) == len(ds[split].unique("Unnamed: 0"))

train
test


This seems to confirm our hypothesis, so let’s clean up the dataset a bit by renaming the Unnamed: 0 column to something a bit more interpretable. We can use the `DatasetDict.rename_column()` function to rename the column across both splits in one go.

In [ ]:
ds = ds.rename_column(original_column_name="Unnamed: 0", new_column_name="patient_id")
ds

DatasetDict({
    train: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount'],
        num_rows: 161297
    })
    test: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount'],
        num_rows: 53766
    })
})

#### Drop Missing Values

In our dataset, some conditions are None . Let’s first drop these rows using `Dataset.filter()`, which works in a similar way to `Dataset.map()` and expects a function that receives a single example of the dataset.

In [ ]:
ds = ds.filter(lambda x: x["condition"] is not None)

Filter:   0%|          | 0/161297 [00:00<?, ? examples/s]

Filter:   0%|          | 0/53766 [00:00<?, ? examples/s]

#### Lowercasing condition

With the None entries removed, we can now normalize our condition column. Lets lowercase the condition.

In [ ]:
def lowercase_condition(example):
    return {"condition": example["condition"].lower()}

ds = ds.map(lowercase_condition)

ds["train"]["condition"][:3]

Map:   0%|          | 0/160398 [00:00<?, ? examples/s]

Map:   0%|          | 0/53471 [00:00<?, ? examples/s]

['left ventricular dysfunction', 'adhd', 'birth control']

#### Review length

Whenever you’re dealing with customer reviews, a good practice is to check the number of words in each review. A review might be just a single word like “Great!” or a full-blown essay with thousands of words, and depending on the use case you’ll need to handle these extremes differently. To compute the number of words in each review, we’ll use a rough heuristic based on splitting each text by whitespace.

Let’s define a simple function that counts the number of words in each review.

In [ ]:
def compute_review_length(example):
    return {"review_length": len(example["review"].split())}

Unlike our `lowercase_condition()` function, `compute_review_length()` returns a dictionary whose key does not correspond to one of the column names in the dataset. In this case, when `compute_review_length()` is passed to `Dataset.map()`, it will be applied to all the rows in the dataset to create a new `review_length` column

In [ ]:
ds = ds.map(compute_review_length)
ds["train"][0]

Map:   0%|          | 0/160398 [00:00<?, ? examples/s]

Map:   0%|          | 0/53471 [00:00<?, ? examples/s]

{'patient_id': 206461,
 'drugName': 'Valsartan',
 'condition': 'left ventricular dysfunction',
 'review': '"It has no side effect, I take it in combination of Bystolic 5 Mg and Fish Oil"',
 'rating': 9.0,
 'date': 'May 20, 2012',
 'usefulCount': 27,
 'review_length': 17}

As expected, we can see a `review_length` column has been added to our training set. We can sort this new column with `Dataset.sort()` to see what the extreme values look like.

In [ ]:
ds["train"].sort("review_length")[:3]

{'patient_id': [111469, 13653, 53602],
 'drugName': ['Ledipasvir / sofosbuvir',
  'Amphetamine / dextroamphetamine',
  'Alesse'],
 'condition': ['hepatitis c', 'adhd', 'birth control'],
 'review': ['"Headache"', '"Great"', '"Awesome"'],
 'rating': [10.0, 10.0, 10.0],
 'date': ['February 3, 2015', 'October 20, 2009', 'November 23, 2015'],
 'usefulCount': [41, 3, 0],
 'review_length': [1, 1, 1]}

In [ ]:
print(ds.num_rows)

{'train': 138514, 'test': 46108}


As we suspected, some reviews contain just a single word, which, although it may be okay for sentiment analysis, would not be informative if we want to predict the condition.

Note: An alternative way to add new columns to a dataset is with the `Dataset.add_column()` function. This allows you to provide the column as a Python list or NumPy array and can be handy in situations where `Dataset.map()` is not well suited for your analysis.

Let’s use the `Dataset.filter()` function to remove reviews that contain fewer than 30 words. Similarly to what we did with the condition column, we can filter out the very short reviews by requiring that the reviews have a length above this threshold.

In [ ]:
ds = ds.filter(lambda example: example["review_length"] > 30)
print(ds.num_rows)

Filter:   0%|          | 0/160398 [00:00<?, ? examples/s]

Filter:   0%|          | 0/53471 [00:00<?, ? examples/s]

{'train': 138514, 'test': 46108}


As you can see, this has removed around 15% of the reviews from our original training and test sets.

#### Escaping HTML chars
The last thing we need to deal with is the presence of HTML character codes in our reviews. We can use Python’s `html` module to unescape these characters, like so.

Basically, we convert all named and numeric character references (e.g. >, >,
&x3e;) in the strings to the corresponding unicode characters.

In [ ]:
import html

text = "I&#039;m a transformer called BERT"
html.unescape(text)

"I'm a transformer called BERT"

In [ ]:
import html
ds = ds.map(lambda x: {"review": html.unescape(x["review"])})

Map:   0%|          | 0/138514 [00:00<?, ? examples/s]

Map:   0%|          | 0/46108 [00:00<?, ? examples/s]

As you can see, the `Dataset.map()` method is quite useful for processing data — and we haven’t even scratched the surface of everything it can do!

#### Batched=True for Faster preprocessing

The `Dataset.map()` method takes a batched argument that, if set to True, causes it to send a batch of examples to the map function at once (the batch size is configurable but defaults to 1,000) which is faster.

When you specify `batched=True` the function receives a `dictionary with the fields of the dataset, but each value is now a list of values, and not just a single value`. The `return value` of `Dataset.map()` should be the `same`: a dictionary with the fields we want to update or add to our dataset, and a list of values.

For eg, here is another way to unescape all HTML characters, but using batched=True

In [ ]:
ds = ds.map(lambda x: {"review": html.unescape(x["review"])}, batched=True)

Map:   0%|          | 0/138514 [00:00<?, ? examples/s]

Map:   0%|          | 0/46108 [00:00<?, ? examples/s]

If you’re running this code in a notebook, you’ll see that this command executes way faster than the previous one. This is because we gain some performance by accessing lots of elements at the same time instead of one by one.

#### Fast tokenizers
Using `Dataset.map()` with `batched=True` will be essential to unlock the speed of the `“fast” tokenizers` that we’ll encounter later, which can quickly tokenize big lists of texts. Let's compare

- slow vs. fast tokenizier
- batched vs no batched
- num_proc vs no num_proc.

In [ ]:
from transformers import AutoTokenizer

# slow tokenizer + batched=True
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased", use_fast=False)
print(tokenizer.is_fast)

def tokenizer_function(examples):
    return tokenizer(examples["review"], truncation=True)

%time tokenized_dsds = ds.map(tokenizer_function, batched=True)



False


Map:   0%|          | 0/138514 [00:00<?, ? examples/s]

Map:   0%|          | 0/46108 [00:00<?, ? examples/s]

CPU times: user 6min 6s, sys: 1.54 s, total: 6min 8s
Wall time: 6min 51s


In [ ]:
from transformers import AutoTokenizer

# slow tokenizer + batched=False
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased", use_fast=False)
print(tokenizer.is_fast)

def tokenizer_function(examples):
    return tokenizer(examples["review"], truncation=True)

%time tokenized_dsds = ds.map(tokenizer_function, batched=False)

False


Map:   0%|          | 0/138514 [00:00<?, ? examples/s]

Map:   0%|          | 0/46108 [00:00<?, ? examples/s]

CPU times: user 8min 9s, sys: 3.94 s, total: 8min 13s
Wall time: 9min 43s


In [ ]:
from transformers import AutoTokenizer

# Fast tokenizer + batched=True
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased", use_fast=True) # default is True
print(tokenizer.is_fast)

def tokenizer_function(examples):
    return tokenizer(examples["review"], truncation=True)

%time tokenized_dsds = ds.map(tokenizer_function, batched=True)

True


Map:   0%|          | 0/138514 [00:00<?, ? examples/s]

Map:   0%|          | 0/46108 [00:00<?, ? examples/s]

CPU times: user 2min 13s, sys: 10.9 s, total: 2min 24s
Wall time: 2min


In [ ]:
from transformers import AutoTokenizer

# Fast tokenizer + batched=False
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased", use_fast=True) # default is True
print(tokenizer.is_fast)

def tokenizer_function(examples):
    return tokenizer(examples["review"], truncation=True)

%time tokenized_dsds = ds.map(tokenizer_function, batched=False)

True


Map:   0%|          | 0/138514 [00:00<?, ? examples/s]

Map:   0%|          | 0/46108 [00:00<?, ? examples/s]

CPU times: user 2min 37s, sys: 1.19 s, total: 2min 39s
Wall time: 2min 53s


In [ ]:
from transformers import AutoTokenizer

# Fast tokenizer + batched=False + num_proc=8
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased", use_fast=True) # default is True
print(tokenizer.is_fast)

def tokenizer_function(examples):
    return tokenizer(examples["review"], truncation=True)

%time tokenized_dsds = ds.map(tokenizer_function, batched=False, num_proc=8)

True


Map (num_proc=8):   0%|          | 0/138514 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/46108 [00:00<?, ? examples/s]

CPU times: user 7.48 s, sys: 1.38 s, total: 8.86 s
Wall time: 2min 53s


A fast tokenizer with the `batched=True` option is 30 times faster than its slow counterpart with no batching which is truly amazing! Fast tokenizer is able to achieve such a speedup because behind the scenes the tokenization code is executed in `Rust`, which is a language that makes it easy to parallelize code execution. Python is the language most often used for data science and deep learning applications, but when anything needs to be parallelized to be fast, it has to be written in another language. For instance, the matrix multiplications that are at the core of the model computation are written in CUDA, an optimized C library for GPUs.

Parallelization is also the reason for the nearly 6x speedup the fast tokenizer achieves with batching: you can’t parallelize a single tokenization operation, but when you want to tokenize lots of texts at the same time you can just split the execution across several processes, each responsible for its own texts.

`Dataset.map()` also has some parallelization capabilities of its own. Since they are not backed by Rust, they won’t let a slow tokenizer catch up with a fast one, but they can still be helpful (especially if you’re using a tokenizer that doesn’t have a fast version). To enable multiprocessing, use the `num_proc` argument and specify the number of processes to use from your CPU in your call to `Dataset.map()`

#### Maximum token length with return_overflowing_tokens=True

Most of the time, pretrained models support certain number of tokens. Here we will tokenize our examples and truncate them to a maximum length of 128, but we will ask the tokenizer to return all the chunks of the texts instead of just the first one. This can be done with `return_overflowing_tokens=True`

In [ ]:
def tokenize_and_split(examples):
    return tokenizer(examples["review"], truncation=True, max_length=128, return_overflowing_tokens=True)

In [ ]:
ds["train"][0]

{'patient_id': 95260,
 'drugName': 'Guanfacine',
 'condition': 'adhd',
 'review': '"My son is halfway through his fourth week of Intuniv. We became concerned when he began this last week, when he started taking the highest dose he will be on. For two days, he could hardly get out of bed, was very cranky, and slept for nearly 8 hours on a drive home from school vacation (very unusual for him.) I called his doctor on Monday morning and she said to stick it out a few days. See how he did at school, and with getting up in the morning. The last two days have been problem free. He is MUCH more agreeable than ever. He is less emotional (a good thing), less cranky. He is remembering all the things he should. Overall his behavior is better. \r\nWe have tried many different medications and so far this is the most effective."',
 'rating': 8.0,
 'date': 'April 27, 2010',
 'usefulCount': 192,
 'review_length': 141}

In [ ]:
# Test on our first example
result = tokenize_and_split(ds["train"][0])
result  # observe we have new column overflow_to_sample_mapping

{'input_ids': [[101, 107, 1422, 1488, 1110, 9079, 1194, 1117, 2223, 1989, 1104, 1130, 19972, 11083, 119, 1284, 1245, 4264, 1165, 1119, 1310, 1142, 1314, 1989, 117, 1165, 1119, 1408, 1781, 1103, 2439, 13753, 1119, 1209, 1129, 1113, 119, 1370, 1160, 1552, 117, 1119, 1180, 6374, 1243, 1149, 1104, 1908, 117, 1108, 1304, 172, 14687, 1183, 117, 1105, 7362, 1111, 2212, 129, 2005, 1113, 170, 2797, 1313, 1121, 1278, 12020, 113, 1304, 5283, 1111, 1140, 119, 114, 146, 1270, 1117, 3995, 1113, 6356, 2106, 1105, 1131, 1163, 1106, 6166, 1122, 1149, 170, 1374, 1552, 119, 3969, 1293, 1119, 1225, 1120, 1278, 117, 1105, 1114, 2033, 1146, 1107, 1103, 2106, 119, 1109, 1314, 1160, 1552, 1138, 1151, 2463, 1714, 119, 1124, 1110, 150, 21986, 3048, 1167, 5340, 1895, 1190, 1518, 102], [101, 119, 1124, 1110, 1750, 6438, 113, 170, 1363, 1645, 114, 117, 1750, 172, 14687, 1183, 119, 1124, 1110, 11566, 1155, 1103, 1614, 1119, 1431, 119, 8007, 1117, 4658, 1110, 1618, 119, 1284, 1138, 1793, 1242, 1472, 23897, 1105, 117

Notice that we got two `input_ids` as list of lists. Let's try to decode what they are.

In [ ]:
tokenizer.decode(result["input_ids"][0])

'[CLS] " My son is halfway through his fourth week of Intuniv. We became concerned when he began this last week, when he started taking the highest dose he will be on. For two days, he could hardly get out of bed, was very cranky, and slept for nearly 8 hours on a drive home from school vacation ( very unusual for him. ) I called his doctor on Monday morning and she said to stick it out a few days. See how he did at school, and with getting up in the morning. The last two days have been problem free. He is MUCH more agreeable than ever [SEP]'

In [ ]:
tokenizer.decode(result["input_ids"][1])

'[CLS]. He is less emotional ( a good thing ), less cranky. He is remembering all the things he should. Overall his behavior is better. We have tried many different medications and so far this is the most effective. " [SEP]'

Now let’s do this for all elements of the dataset!

In [ ]:
# this will give error
tokenized_ds = ds.map(tokenize_and_split, batched=True)

Map:   0%|          | 0/138514 [00:00<?, ? examples/s]

ArrowInvalid: Column 8 named input_ids expected length 1000 but got length 1463

Oh no! That didn’t work! Why not? Looking at the error message will give us a clue:

For our dataset, there is a mismatch in the lengths of one of the columns, one being of length 1,463 and the other of length 1,000. If you’ve looked at the `Dataset.map()` documentation, `1000` is the `default batch size` of `Dataset.map()`, while `1463` comes from chunking.


The code `tokenized_ds = ds.map(tokenize_and_split, batched=True)` will throw an error when `return_overflowing_tokens=True` is used with `batched=True` in the `map` function.

The core issue is a **mismatch in the number of samples (rows) returned for different features within a batch.**

Here's why:

1.  **`batched=True`:** When `batched=True`, the `map` function receives a dictionary where each key corresponds to a column, and the value is a *list* of elements for that column, representing a batch of examples. For instance, `examples["review"]` will be a list of review strings. The `map` function expects to return a dictionary where each key's value is a list of the *same length* as the input lists for that batch. This preserves the row-wise alignment of your dataset.

2.  **`return_overflowing_tokens=True`:** This tokenizer parameter is designed to handle long texts by splitting them into multiple chunks that fit within `max_length`. If a single original review is very long, it might be tokenized into 2, 3, or even more separate tokenized chunks (each with `max_length` or less).

3.  **The Mismatch:**

      * If you have a batch of, say, 10 reviews.
      * One review is short and gets tokenized into 1 chunk(as a list).
      * Another review is very long and gets tokenized into 3 chunks(3 lists per se).
      * The `tokenizer` with `return_overflowing_tokens=True` will return a flattened list of all these chunks.
      * So, for your `input_ids` column, you might end up with `1 + 3 = 4` tokenized sequences just from those two reviews. If all other 8 reviews also produce just 1 chunk each, your output batch for `input_ids` would have `4 + 8 = 12` items.
      * However, the original `examples["review"]` list only had 10 items. The `datasets` library tries to create a new dataset where the number of rows in the `input_ids` column (and `attention_mask`, etc.) matches the number of rows in the original `review` column. Since `12 != 10`, you get an `ArrowInvalid` error (or similar) because the lengths of the transformed columns don't match the original batch size.

**Which lengths are not matching?**

The lengths of the lists (representing the number of examples/rows) for the *output columns* of your `tokenize_and_split` function are not matching the number of examples in the *input batch*.

Specifically:

  * Input batch size: `len(examples["review"])`
  * Output batch size for `input_ids`, `attention_mask`, etc.: This will be `sum(num_chunks_for_each_original_review_in_batch)`.

Since `sum(num_chunks_for_each_original_review_in_batch)` can be greater than `len(examples["review"])`, the `datasets` library cannot correctly align the new features with the original features in a 1:1 mapping.

### How to Fix It:

There are a few common ways to handle this, depending on your goal:

1.  **Process without `batched=True` (and potentially slower):**
    If your dataset isn't too large, you could remove `batched=True` from the `map` call.

    ```python
    def tokenize_and_split(example): # 'example' is now a single dictionary
        return tokenizer(example["review"], truncation=True, max_length=128, return_overflowing_tokens=True)

    tokenized_ds = ds.map(tokenize_and_split) # No batched=True
    ```

    However, this often makes tokenization much slower as it loses the speed benefits of batching.

2.  **Process in batches, but handle the "overflow" explicitly:**
    The recommended way is to collect all the overflowing tokens and map them back to the original examples, usually by creating a new, "flattened" dataset. You'll also typically want to store `overflow_to_sample_mapping` and `offset_mapping` to know which original sample each new chunk came from.

    ```python
    def tokenize_and_split_for_long_docs(examples):
        # Tokenize the batch
        tokenized_inputs = tokenizer(
            examples["review"],
            truncation=True,
            max_length=128,
            return_overflowing_tokens=True,
            return_offsets_mapping=True, # Useful for re-mapping
        )

        # Prepare new lists to hold the flattened results
        result = {
            "input_ids": [],
            "attention_mask": [],
            "token_type_ids": [], # Include if your tokenizer uses them
            "labels": [] # If you have labels, they also need to be handled
        }
        
        # Keep track of which original sample each new chunk belongs to
        sample_mapping = tokenized_inputs.pop("overflow_to_sample_mapping")
        offset_mapping = tokenized_inputs.pop("offset_mapping")

        # Iterate over the new chunks created by `return_overflowing_tokens`
        for i, input_ids in enumerate(tokenized_inputs["input_ids"]):
            # Add tokenized outputs for this chunk
            result["input_ids"].append(input_ids)
            result["attention_mask"].append(tokenized_inputs["attention_mask"][i])
            if "token_type_ids" in tokenized_inputs:
                result["token_type_ids"].append(tokenized_inputs["token_type_ids"][i])

            # If you have labels, you need to align them to the new chunks.
            # This is often the trickiest part and depends on your task.
            # For a simple classification task where each *document* has one label,
            # you'd assign the original label to each chunk from that document.
            original_sample_idx = sample_mapping[i]
            if "label" in examples: # Assuming your dataset has a 'label' column
                result["labels"].append(examples["label"][original_sample_idx])
            # If your labels are token-level (e.g., NER), this becomes more complex
            # and requires aligning tokens from the overflowing chunks.

        return result

    # This will create a new dataset where one original review might correspond to multiple rows.
    tokenized_ds = ds.map(
        tokenize_and_split_for_long_docs,
        batched=True,
        remove_columns=ds.column_names # Remove original columns that won't match row count
    )
    ```

    This approach is common for tasks like question answering or long document classification where you need to process full texts that exceed the model's `max_length`.

3. We can also either remove the columns from the original dataset(which is not mathcing) with the `remove_columns` argument.

In [ ]:
ds

DatasetDict({
    train: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 138514
    })
    test: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 46108
    })
})

In [ ]:
len(ds["train"]["review"])

138514

In [ ]:
ds["train"].column_names

['patient_id',
 'drugName',
 'condition',
 'review',
 'rating',
 'date',
 'usefulCount',
 'review_length']

In [ ]:
ds["train"].unique("drugName")

In [ ]:
# option 3: Remove columns from the original dataset for our function to work
tokenized_ds = ds.map(tokenize_and_split, batched=True, remove_columns=ds["train"].column_names)  # Removes original columns that won't match row count

Map:   0%|          | 0/138514 [00:00<?, ? examples/s]

Map:   0%|          | 0/46108 [00:00<?, ? examples/s]

Now this works without error. We can check that our new dataset has many more elements than the original dataset by comparing the lengths.

In [ ]:
tokenized_ds

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'overflow_to_sample_mapping'],
        num_rows: 206772
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'overflow_to_sample_mapping'],
        num_rows: 68876
    })
})

In [ ]:
tokenized_ds["train"].features

{'input_ids': Sequence(feature=Value(dtype='int32', id=None), length=-1, id=None),
 'token_type_ids': Sequence(feature=Value(dtype='int8', id=None), length=-1, id=None),
 'attention_mask': Sequence(feature=Value(dtype='int8', id=None), length=-1, id=None),
 'overflow_to_sample_mapping': Value(dtype='int64', id=None)}

In [ ]:
tokenized_ds["train"].features["input_ids"]

Now this works without error. We can check that our new dataset has many more elements than the original dataset by comparing the lengths.

In [ ]:
len(ds["train"]), len(tokenized_ds["train"])

(138514, 206772)

option 2:

We mentioned that we can also deal with the mismatched length problem by making the old columns the same size as the new ones. To do this, we will need the `overflow_to_sample_mapping` field the tokenizer returns when we set `return_overflowing_tokens=True`.

It gives us a mapping from a new feature index to the index of the sample it originated from. Using this, we can associate each key present in our original dataset with a list of values of the right size by repeating the values of each example as many times as it generates new features.

In [ ]:
def tokenize_and_split(examples):
    result = tokenizer(examples["review"], truncation=True, max_length=128, return_overflowing_tokens=True)
    # Extract mapping between new and original review indices
    mapping = result.pop("overflow_to_sample_mapping")
    for k, v in examples.items(): # access original data
        result[k] = [v[idx] for idx in mapping]
    return result

In [ ]:
tokenized_ds = ds.map(tokenize_and_split, batched=True)
tokenized_ds

Map:   0%|          | 0/138514 [00:00<?, ? examples/s]

Map:   0%|          | 0/46108 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 206772
    })
    test: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 68876
    })
})

We get the same number of training features as before, but here we’ve kept all the old fields. If you need them for some post-processing after applying your model, this is the recommended approach.



#### Using Pandas with Datasets Dataset.set_format("pandas") and Dataset.from_pandas(df)
You’ve now seen how 🤗 Datasets can be used to preprocess a dataset in various ways. Although the processing functions of 🤗 Datasets will cover most of your model training needs, there may be times when you’ll need to switch to Pandas to access more powerful features, like `DataFrame.groupby()` or high-level APIs for visualization. Fortunately, 🤗 Datasets is designed to be interoperable with libraries such as Pandas, NumPy, PyTorch, TensorFlow, and JAX. Let’s take a look at how this works.

To enable the conversion between various third-party libraries, 🤗 Datasets provides a `Dataset.set_format()` function.

Under the hood, `Dataset.set_format()` changes the return format for the dataset’s `getitem()` dunder method. This means that when we want to create a new object like `train_df` from a Dataset in the "pandas" format, we need to slice the whole dataset to obtain a `pandas.DataFrame`. You can verify for yourself that the `type of dataset["train"]` is `Dataset`, irrespective of the output format.

This function only changes the output format of the dataset, so you can easily switch to another format without affecting the underlying data format, which is Apache Arrow. The formatting is done in place. To demonstrate, let’s convert our dataset to Pandas.

In [ ]:
type(ds)

datasets.dataset_dict.DatasetDict

In [ ]:
type(ds["train"])

datasets.arrow_dataset.Dataset

In [ ]:
type(ds["train"][:3])

dict

In [ ]:
ds.set_format("pandas") # inplace

In [ ]:
type(ds)

datasets.dataset_dict.DatasetDict

In [ ]:
type(ds["train"])

datasets.arrow_dataset.Dataset

In [ ]:
type(ds["train"][:3])

pandas.core.frame.DataFrame

In [ ]:
ds["train"][:5]

,patient_id,drugName,condition,review,rating,date,usefulCount,review_length
0,95260,Guanfacine,adhd,"""My son is halfway through his fourth week of ...",8.0,"April 27, 2010",192,141
1,92703,Lybrel,birth control,"""I used to take another oral contraceptive, wh...",5.0,"December 14, 2009",17,134
2,138000,Ortho Evra,birth control,"""This is my first time using any form of birth...",8.0,"November 3, 2015",10,89
3,35696,Buprenorphine / naloxone,opiate dependence,"""Suboxone has completely turned my life around...",9.0,"November 27, 2016",37,124
4,155963,Cialis,benign prostatic hyperplasia,"""2nd day on 5mg started to work with rock hard...",2.0,"November 28, 2015",43,68


Let’s create a pandas.DataFrame for the whole training set by selecting all the elements of drug_dataset["train"]

In [ ]:
train_df = ds["train"][:]

From here we can use all the Pandas functionality that we want.

In [ ]:
train_df.head()

,patient_id,drugName,condition,review,rating,date,usefulCount,review_length
0,95260,Guanfacine,adhd,"""My son is halfway through his fourth week of ...",8.0,"April 27, 2010",192,141
1,92703,Lybrel,birth control,"""I used to take another oral contraceptive, wh...",5.0,"December 14, 2009",17,134
2,138000,Ortho Evra,birth control,"""This is my first time using any form of birth...",8.0,"November 3, 2015",10,89
3,35696,Buprenorphine / naloxone,opiate dependence,"""Suboxone has completely turned my life around...",9.0,"November 27, 2016",37,124
4,155963,Cialis,benign prostatic hyperplasia,"""2nd day on 5mg started to work with rock hard...",2.0,"November 28, 2015",43,68


And once we’re done with our Pandas analysis, we can always create a new Dataset object by using the `Dataset.from_pandas()` function as follows.

In [ ]:
from datasets import Dataset

train_ds = Dataset.from_pandas(train_df)
train_ds

Dataset({
    features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
    num_rows: 138514
})

In [ ]:
### set format using different kwargs
format = {'type': 'torch', 'format_kwargs' :{'dtype': torch.long}}
dataset.set_format(**format)

#### Reset format back to Dataset ds.reset_format()
To round out the section, let’s create a validation set to prepare the dataset for training a classifier on. Before doing so, we’ll reset the output format of drug_dataset from "pandas" to "arrow" using `ds.reset_format()`

In [ ]:
ds.reset_format()

In [ ]:
ds

DatasetDict({
    train: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 138514
    })
    test: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 46108
    })
})

#### Creating validation set using train_test_split
Although we have a test set we could use for evaluation, it’s a good practice to leave the test set untouched and create a separate validation set during development. Once you are happy with the performance of your models on the validation set, you can do a final sanity check on the test set. This process helps mitigate the risk that you’ll overfit to the test set and deploy a model that fails on real-world data.

🤗 Datasets provides a `Dataset.train_test_split()` function that is based on the famous functionality from scikit-learn. Let’s use it to split our training set into train and validation splits (we set the seed argument for reproducibility).

In [ ]:
ds_clean = ds["train"].train_test_split(train_size=0.8, seed=42) # creates train and test set from our selected train data
ds_clean

DatasetDict({
    train: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 110811
    })
    test: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 27703
    })
})

In [ ]:
# Rename the default "test" split to "validation"
ds_clean["validation"] = ds_clean.pop("test")
ds_clean

DatasetDict({
    train: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 110811
    })
    validation: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 27703
    })
})

In [ ]:
# Add the "test" set to our `DatasetDict`
ds_clean["test"] = ds["test"]
ds_clean

DatasetDict({
    train: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 110811
    })
    validation: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 27703
    })
    test: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 46108
    })
})

#### Saving a dataset
Finally, after preprocessing, we may want to save the Dataset. Huggingface provides `Dataset.save_to_disk()` (Apache Arrow), `Dataset.to_csv` (csv) and `Dataset.to_json()` (JSON)

For example, let’s save our cleaned dataset in the Arrow format.

In [ ]:
ds_clean.save_to_disk("drug_reviews")

Saving the dataset (0/1 shards):   0%|          | 0/110811 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/27703 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/46108 [00:00<?, ? examples/s]

Notice you got some new dir. We can load it by using the `load_from_disk()` function as follows

In [ ]:
from datasets import load_from_disk

ds_reloaded = load_from_disk("drug_reviews")
ds_reloaded

DatasetDict({
    train: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 110811
    })
    validation: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 27703
    })
    test: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 46108
    })
})

For the CSV and JSON formats, we have to store each split as a separate file. One way to do this is by iterating over the keys and values in the DatasetDict object.

In [ ]:
for split, ds in ds_clean.items():
    ds.to_json(f"drug-reviews-{split}.jsonl")

Creating json from Arrow format:   0%|          | 0/111 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/28 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/47 [00:00<?, ?ba/s]

This saves each split in JSON Lines format, where each row in the dataset is stored as a single line of JSON. Here’s what the first example looks like

In [ ]:
!head -n 1 drug-reviews-train.jsonl

{"patient_id":89879,"drugName":"Cyclosporine","condition":"keratoconjunctivitis sicca","review":"\"I have used Restasis for about a year now and have seen almost no progress.  For most of my life I've had red and bothersome eyes. After trying various eye drops, my doctor recommended Restasis.  He said it typically takes 3 to 6 months for it to really kick in but it never did kick in.  When I put the drops in it burns my eyes for the first 30 - 40 minutes.  I've talked with my doctor about this and he said it is normal but should go away after some time, but it hasn't. Every year around spring time my eyes get terrible irritated  and this year has been the same (maybe even worse than other years) even though I've been using Restasis for a year now. The only difference I notice was for the first couple weeks, but now I'm ready to move on.\"","rating":2.0,"date":"April 20, 2013","usefulCount":69,"review_length":147}


We can load the json file using `load_dataset`

In [ ]:
from datasets import load_dataset

data_files = {
    "train": "drug-reviews-train.jsonl",
    "validation": "drug-reviews-validation.jsonl",
    "test": "drug-reviews-test.jsonl",
}
ds_reloaded  = load_dataset("json", data_files=data_files)
ds_reloaded

DatasetDict({
    train: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 110811
    })
    validation: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 27703
    })
    test: Dataset({
        features: ['patient_id', 'drugName', 'condition', 'review', 'rating', 'date', 'usefulCount', 'review_length'],
        num_rows: 46108
    })
})